# Calibration Engine Research Demo

This notebook demonstrates a reproducible parameter calibration workflow using synthetic data only. It is domain-neutral and runs fully offline.

The goal is to show a research-style loop:

1. define a controlled data-generating process;
2. define an objective function;
3. define a validated search space;
4. run calibration with metadata;
5. inspect top trials, summaries and persisted results.

## Setup

The notebook uses only `numpy`, `pandas` and the local `calibration_engine` package.

In [ ]:
import numpy as np
import pandas as pd

from calibration_engine import CalibrationEngine, CalibrationResult, SearchSpace

## Synthetic Experiment

We create a noisy linear system where the true process is known. This allows the calibration run to be inspected without requiring private data or external services.

In [ ]:
rng = np.random.default_rng(42)
x = np.linspace(-3, 3, 180)
noise = rng.normal(0, 0.35, size=len(x))
y = 1.75 * x - 0.40 + noise

data = pd.DataFrame({"x": x, "y": y})
data.head()

## Objective Function

The objective returns a scalar `score` plus extra metrics. The engine maximizes `score`, so we use negative RMSE.

In [ ]:
def objective(params, data):
    prediction = params["slope"] * data["x"] + params["intercept"]
    error = data["y"] - prediction
    rmse = float(np.sqrt(np.mean(error**2)))
    mae = float(np.mean(np.abs(error)))
    return {"score": -rmse, "rmse": rmse, "mae": mae}

## Search Space

`SearchSpace` validates the parameter space and serializes a manifest into the result metadata.

In [ ]:
space = SearchSpace.from_dict({
    "slope": (0.0, 3.0),
    "intercept": (-2.0, 2.0),
})

space.to_dict()

## Random Search

Random search is a strong baseline for many calibration problems. The seed makes the run reproducible.

In [ ]:
random_result = CalibrationEngine(seed=123).calibrate(
    objective,
    param_space=space,
    data=data,
    optimizer="random",
    max_evals=250,
    direction="maximize",
    metadata={"experiment": "synthetic-linear-random"},
)

random_result.best_params, random_result.best_metrics

## Grid Search

Grid search is deterministic and useful for small spaces or sanity checks.

In [ ]:
grid_result = CalibrationEngine(seed=123).calibrate(
    objective,
    param_space=space,
    data=data,
    optimizer="grid",
    max_evals=100,
    direction="maximize",
    metadata={"experiment": "synthetic-linear-grid"},
)

grid_result.best_params, grid_result.best_metrics

## Audit Tables

Every candidate is preserved. This is the core audit trail.

In [ ]:
random_result.to_frame().sort_values("score", ascending=False).head(10)

## Summary and Report

The result object provides compact summaries and a Markdown report.

In [ ]:
random_result.summary()

In [ ]:
print(random_result.report(top_n=3))

## Local Persistence

Results can be saved locally as JSON and trial tables as CSV. No network or external service is involved.

In [ ]:
output_dir = "../examples/data"
random_result.save_json(f"{output_dir}/notebook_result.json")
random_result.save_csv(f"{output_dir}/notebook_trials.csv")

loaded = CalibrationResult.load_json(f"{output_dir}/notebook_result.json")
loaded.best_params

## Interpretation

The framework separates research mechanics from model logic. The objective function owns the domain logic; Calibration Engine owns search, trial accounting, summaries, metadata and persistence.